## Launching model on dataset and collecting output

### Imports

In [20]:
from datasets import load_dataset
from tqdm import tqdm

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from enum import Enum

import torch
import sys

sys.path.append("../../services/")
from index import Index
from process_funcs import retrieve_answer_token_index

torch.random.manual_seed(42)


ModuleNotFoundError: No module named 'index'

### Preparation

In [ ]:
ITERATIONS_COUNT = 12000

In [ ]:
PossibleOutput = Enum(
    "PossibleOutput", {f"PossibleValue{i}": str(i) for i in range(10)}
)


class OutputTemplate(BaseModel):
    short_chain_of_thoughts: str = Field(description="Short chain of thoughts")
    answer: PossibleOutput = Field(  # type: ignore
        description="One number of answer variant",
    )

In [ ]:
dataset = load_dataset("TIGER-Lab/MMLU-Pro", split="test")

chat_llm = ChatOpenAI(
    model="llama3.1:8b",
    api_key="",
    base_url="http://localhost:11101/v1",
    max_tokens=4096,
    logprobs=True,
    top_logprobs=20,
)

parser = PydanticOutputParser(pydantic_object=OutputTemplate)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are an expert at answering multiple choice questions. 
            You will be given a question and options. Follow these rules strictly:
            
            1. Think step by step, provide chain of thoughts leading to the answer
            2. Select the correct option number (0 - 9)
            3. Return your response in this EXACT JSON format:
            
            {format_instructions}
            
            Do not include any additional text outside the JSON format.
            Your response must be valid JSON that can be parsed automatically.
            Answer following question:
            """,
        ),
        (
            "human",
            """
            Question: {input_question}
    
            Options:
            {input_options}
            """,
        ),
    ]
)

llm_chain = prompt | chat_llm

### Running model on dataset and collecting results


In [ ]:
def filter_func(model_response):
    """Return True/False whether answer token distribution does have all 0-9 tokens"""
    log = model_response.response_metadata["logprobs"]["content"][
        retrieve_answer_token_index(
            model_response.response_metadata["logprobs"]["content"]
        )
    ]
    if (
        len(
            set([str(x) for x in range(10)]).intersection(
                set([x["token"] for x in log["top_logprobs"]])
            )
        )
        < 10
    ):
        return False
    return True

In [ ]:
# launching model

ERROR_LIMIT = 1000

index = Index(f"../index_data/llama3.1:8b_HaluEval_launch_{ITERATIONS_COUNT}")
index.clear()

correct_count = 0
progress_bar = tqdm(
    enumerate(dataset.select(range(ITERATIONS_COUNT))),
    total=ITERATIONS_COUNT,
    desc="Scoring HaluEval",
)

for iter, data in progress_bar:
    for error_count in range(ERROR_LIMIT + 1):
        try:
            response = llm_chain.invoke(
                {
                    "format_instructions": parser.get_format_instructions(),
                    "input_question": dataset[iter]["question"],
                    "input_options": [
                        f"{i}:  {x}"
                        for i, x in enumerate(dataset[iter]["options"])
                    ],
                }
            )

            if parser.parse(response.content) and filter_func(response):
                result = {
                    "iteration": iter,
                    "text": response.content,
                }

                if parser.parse(response.content).answer.value == str(
                    data["answer_index"]
                ):
                    correct_count += 1

                progress_bar.set_postfix(
                    {"Current accuracy": str(correct_count / (iter + 1))}
                )

                result = {
                    "iteration": iter,
                    "text": response.content,
                    "logprobs": response.response_metadata["logprobs"][
                        "content"
                    ],
                }

                index.save_data(result, iter, logging=False)
                break

        except Exception as e:
            if error_count == ERROR_LIMIT:
                print("Errors limit exceeded")
                print(e)

    # print(data["answer_index"] == extract_predicted_index(text[-30:]))
    # print(extract_predicted_index(text[-30:]))
    # print("\n\n")

Файл не существует: ../index_data/llama3.1:8b_MMLU-PRO_launch_12000_index.pkl
Файл не существует: ../index_data/llama3.1:8b_MMLU-PRO_launch_12000_data.pkl
Index is cleared successfully.


Scoring MMLU-PRO:   0%|          | 0/12000 [00:00<?, ?it/s]

Scoring MMLU-PRO:  34%|███▍      | 4067/12000 [9:44:45<19:00:36,  8.63s/it, Current accuracy=0.3567740349151709]  


KeyboardInterrupt: 

In [ ]:
print(len(index))